# 💳 Credit Card Fraud Detection Using Machine Learning
**Group 2 – CSC Project**  
Department of Computer Science, Federal University of Technology Owerri  
Supervisor: Dr. Jacinta Chioma Odirichukwu

---

## Project Overview
This notebook develops a machine learning-based fraud detection system that identifies fraudulent credit card transactions using historical transaction data. It covers:

1. Data Collection & Loading  
2. Data Cleaning & Preprocessing  
3. Handling Class Imbalance (SMOTE)  
4. Exploratory Data Analysis (EDA)  
5. Feature Selection  
6. Model Building  
7. Model Evaluation  
8. Model Comparison  
9. Prediction System Development  
10. Result Interpretation  

**Dataset:** [Kaggle Credit Card Fraud Dataset](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

---
## Step 1 – Imports & Setup
We import all required libraries:
- **Pandas & NumPy** for data manipulation
- **Matplotlib & Seaborn** for visualisation
- **Scikit-learn** for machine learning models and evaluation
- **Imbalanced-learn** for SMOTE oversampling

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, f1_score, precision_score, recall_score
)
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")
print("All libraries imported successfully.")

---
## Step 2 – Data Collection & Loading
The dataset contains transactions made by European cardholders in September 2013. It has **284,807 transactions** with only **492 frauds** (0.172%), making it highly imbalanced.

Features V1–V28 are PCA-transformed for confidentiality. `Time`, `Amount`, and `Class` are the original features.

In [ ]:
BASE_DIR  = os.path.dirname(os.path.abspath("__file__"))
DATA_PATH = os.path.join(BASE_DIR, "creditcard.csv")
PLOT_DIR  = os.path.join(BASE_DIR, "plots")
MODEL_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(PLOT_DIR,  exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape  : {df.shape}")
print(f"Missing values : {df.isnull().sum().sum()}")
df.head()

In [ ]:
print("Data Types:")
print(df.dtypes)
print("\nBasic Statistics:")
df.describe()

In [ ]:
counts = df["Class"].value_counts()
print("Class Distribution:")
print(f"  Legitimate (0) : {counts[0]:,}  ({counts[0]/len(df)*100:.2f}%)")
print(f"  Fraudulent (1) : {counts[1]:,}  ({counts[1]/len(df)*100:.4f}%)")

---
## Step 3 – Exploratory Data Analysis (EDA)
We visualise the class distribution, transaction amounts, fraud patterns by hour, and feature correlations.

In [ ]:
# Class distribution & amount distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

counts = df["Class"].value_counts()
axes[0].bar(["Legitimate", "Fraudulent"], counts.values,
            color=["steelblue", "crimson"], edgecolor="black")
axes[0].set_title("Class Distribution")
axes[0].set_ylabel("Number of Transactions")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 500, f"{v:,}", ha="center", fontweight="bold")

axes[1].hist(df[df["Class"]==0]["Amount"], bins=60, alpha=0.6,
             label="Legitimate", color="steelblue")
axes[1].hist(df[df["Class"]==1]["Amount"], bins=60, alpha=0.8,
             label="Fraudulent", color="crimson")
axes[1].set_title("Transaction Amount Distribution")
axes[1].set_xlabel("Amount (USD)")
axes[1].set_ylabel("Count")
axes[1].legend()
axes[1].set_xlim(0, 2500)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "01_class_and_amount.png"), dpi=150)
plt.show()

In [ ]:
# Fraud rate by hour of day
df_copy = df.copy()
df_copy["Hour"] = (df_copy["Time"] // 3600) % 24
hourly = df_copy.groupby("Hour")["Class"].mean() * 100

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(hourly.index, hourly.values, marker="o", color="crimson", linewidth=2)
ax.fill_between(hourly.index, hourly.values, alpha=0.15, color="crimson")
ax.set_title("Fraud Rate by Hour of Day")
ax.set_xlabel("Hour")
ax.set_ylabel("Fraud Rate (%)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "03_fraud_by_hour.png"), dpi=150)
plt.show()

In [ ]:
# Correlation heatmap – top 15 features vs Class
corr = df.corr()["Class"].drop("Class").abs().sort_values(ascending=False)
top_feats = corr.head(15).index.tolist()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(df[top_feats + ["Class"]].corr(), annot=True, fmt=".2f",
            cmap="coolwarm", ax=ax, linewidths=0.5)
ax.set_title("Correlation Heatmap – Top 15 Features vs Class")
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "02_correlation_heatmap.png"), dpi=150)
plt.show()

In [ ]:
# Box plots: top 6 features by fraud vs legitimate
top6 = corr.head(6).index.tolist()
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, feat in zip(axes.flatten(), top6):
    df.boxplot(column=feat, by="Class", ax=ax, 
               boxprops=dict(color="steelblue"),
               medianprops=dict(color="crimson", linewidth=2))
    ax.set_title(feat)
    ax.set_xlabel("Class (0=Legit, 1=Fraud)")
plt.suptitle("Top 6 Features: Legitimate vs Fraudulent", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Step 4 – Feature Selection
We rank features by their absolute correlation with the `Class` label and identify the most discriminative ones.

In [ ]:
corr_all = df.corr()["Class"].drop("Class").abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 9))
corr_all.sort_values().plot.barh(ax=ax, color="steelblue", edgecolor="black")
ax.set_title("Feature Correlation with Fraud (absolute)")
ax.set_xlabel("|Pearson r|")
plt.tight_layout()
plt.show()

print("Top 10 features most correlated with fraud:")
print(corr_all.head(10).to_string())

---
## Step 5 – Data Cleaning & Preprocessing
- Scale `Amount` and `Time` using `StandardScaler` (V1–V28 are already PCA-scaled)
- Split into 80% train / 20% test with stratification to preserve class ratios

In [ ]:
scaler = StandardScaler()
df_proc = df.copy()
df_proc["Amount"] = scaler.fit_transform(df_proc[["Amount"]])
df_proc["Time"]   = scaler.fit_transform(df_proc[["Time"]])

X = df_proc.drop("Class", axis=1)
y = df_proc["Class"]
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size : {X_train.shape[0]:,}")
print(f"Test size  : {X_test.shape[0]:,}")
print(f"\nTrain class distribution:")
print(y_train.value_counts().to_string())

---
## Step 6 – Handling Class Imbalance
We use **SMOTE** (Synthetic Minority Oversampling Technique) to generate synthetic fraud samples in the training set, balancing both classes without touching the test set.

We also create an undersampled version for comparison.

In [ ]:
# SMOTE oversampling
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train, y_train)
print("After SMOTE:")
print(f"  Legitimate : {(y_res==0).sum():,}")
print(f"  Fraudulent : {(y_res==1).sum():,}")

# Random undersampling
rus = RandomUnderSampler(random_state=42)
X_under, y_under = rus.fit_resample(X_train, y_train)
print("\nAfter Random Undersampling:")
print(f"  Legitimate : {(y_under==0).sum():,}")
print(f"  Fraudulent : {(y_under==1).sum():,}")

In [ ]:
# Visualise class balance before and after SMOTE
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
labels = ["Legitimate", "Fraudulent"]
colors = ["steelblue", "crimson"]

datasets = [
    ("Original Train",      y_train),
    ("After SMOTE",         y_res),
    ("After Undersampling", y_under),
]
for ax, (title, y_) in zip(axes, datasets):
    vals = [( y_==0).sum(), (y_==1).sum()]
    ax.bar(labels, vals, color=colors, edgecolor="black")
    ax.set_title(title)
    ax.set_ylabel("Count")
    for i, v in enumerate(vals):
        ax.text(i, v + max(vals)*0.01, f"{v:,}", ha="center", fontsize=9)

plt.suptitle("Class Balance: Before vs After Resampling", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Step 7 – Model Building
We train three machine learning models on the SMOTE-balanced training data:
1. **Logistic Regression** – linear baseline
2. **Decision Tree** – interpretable tree-based model
3. **Random Forest** – ensemble of decision trees

In [ ]:
models_def = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree":       DecisionTreeClassifier(max_depth=8, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}

results = []
trained = {}
preds   = {}

for name, model in models_def.items():
    model.fit(X_res, y_res)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    precision = precision_score(y_test, y_pred)
    recall    = recall_score(y_test, y_pred)
    f1        = f1_score(y_test, y_pred)
    roc_auc   = roc_auc_score(y_test, y_proba)

    trained[name] = model
    preds[name]   = (y_pred, y_proba)
    results.append({"Model": name, "Precision": precision,
                    "Recall": recall, "F1": f1, "ROC_AUC": roc_auc})

    print(f"[{name}]")
    print(f"  Precision : {precision:.4f}  Recall : {recall:.4f}  F1 : {f1:.4f}  ROC-AUC : {roc_auc:.4f}")
    print()

df_res = pd.DataFrame(results)
print(df_res.to_string(index=False))

---
## Step 8 – Model Evaluation
We evaluate each model using classification reports, confusion matrices, ROC curves, and Precision-Recall curves.

In [ ]:
# Classification reports
for name, (y_pred, _) in preds.items():
    print(f"=== {name} ===")
    print(classification_report(y_test, y_pred,
                                target_names=["Legitimate", "Fraudulent"],
                                digits=4))

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, (y_pred, _)) in zip(axes, preds.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Legit", "Fraud"],
                yticklabels=["Legit", "Fraud"])
    ax.set_title(name)
    ax.set_ylabel("Actual")
    ax.set_xlabel("Predicted")
plt.suptitle("Confusion Matrices", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "07_confusion_matrices.png"), dpi=150)
plt.show()

In [ ]:
# ROC curves
colors_list = ["#4C72B0", "#DD8452", "#55A868"]
fig, ax = plt.subplots(figsize=(8, 6))
for (name, (_, y_proba)), color in zip(preds.items(), colors_list):
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.4f})", color=color, lw=2)
ax.plot([0,1],[0,1], "k--", lw=1)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves – All Models")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "05_roc_curves.png"), dpi=150)
plt.show()

In [ ]:
# Precision-Recall curves
fig, ax = plt.subplots(figsize=(8, 6))
for (name, (_, y_proba)), color in zip(preds.items(), colors_list):
    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    ax.plot(rec, prec, label=name, color=color, lw=2)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves – All Models")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "06_precision_recall_curves.png"), dpi=150)
plt.show()

---
## Step 9 – Model Comparison
We compare all three models side-by-side across Precision, Recall, F1-Score, and ROC-AUC.

In [ ]:
metrics = ["Precision", "Recall", "F1", "ROC_AUC"]
x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
for i, (_, row) in enumerate(df_res.iterrows()):
    ax.bar(x + i * width, [row[m] for m in metrics],
           width, label=row["Model"], color=colors_list[i], alpha=0.85, edgecolor="black")

ax.set_xticks(x + width)
ax.set_xticklabels(["Precision", "Recall", "F1-Score", "ROC-AUC"])
ax.set_ylim(0, 1.15)
ax.set_ylabel("Score")
ax.set_title("Model Performance Comparison")
ax.legend()
ax.grid(axis="y", alpha=0.3)

for bars in ax.containers:
    ax.bar_label(bars, fmt="%.3f", fontsize=8, padding=2)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "04_model_comparison.png"), dpi=150)
plt.show()

print("\nFull comparison table:")
print(df_res.to_string(index=False))

In [ ]:
# Cross-validation for robustness check
print("5-Fold Stratified Cross-Validation (F1-Score) on SMOTE data:\n")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, model in trained.items():
    scores = cross_val_score(model, X_res, y_res, cv=cv, scoring="f1", n_jobs=-1)
    print(f"  {name:<22} : {scores.mean():.4f} ± {scores.std():.4f}")

---
## Step 10 – Feature Importance & Result Interpretation
Using Random Forest's built-in feature importances, we identify which features are most predictive of fraud.

In [ ]:
rf = trained["Random Forest"]
importances = pd.Series(rf.feature_importances_, index=feature_names)
top15 = importances.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 5))
top15.sort_values().plot.barh(ax=ax, color="steelblue", edgecolor="black")
ax.set_title("Top 15 Feature Importances – Random Forest")
ax.set_xlabel("Importance Score")
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "08_feature_importance.png"), dpi=150)
plt.show()

print("Top 10 Most Important Features:")
print(top15.head(10).to_string())

In [ ]:
# Select and save best model
best_row   = df_res.loc[df_res["F1"].idxmax()]
best_name  = best_row["Model"]
best_model = trained[best_name]

print(f"Best model (by F1-Score): {best_name}")
print(f"  F1-Score : {best_row['F1']:.4f}")
print(f"  ROC-AUC  : {best_row['ROC_AUC']:.4f}")

model_path = os.path.join(MODEL_DIR, "best_model.pkl")
joblib.dump(best_model, model_path)
print(f"  Saved to : {model_path}")

---
## Step 11 – Prediction System
A simple prediction function that takes a transaction and returns a fraud probability and label.

In [ ]:
def predict_transaction(model, transaction_dict, feature_names):
    row  = pd.DataFrame([transaction_dict], columns=feature_names)
    prob = model.predict_proba(row)[0][1]
    label = "FRAUDULENT" if prob >= 0.5 else "LEGITIMATE"
    return label, prob

# Demo: one known fraud, one known legitimate transaction
fraud_idx = y_test[y_test == 1].index[0]
legit_idx = y_test[y_test == 0].index[0]

for idx, true_label in [(fraud_idx, "FRAUDULENT"), (legit_idx, "LEGITIMATE")]:
    txn = X_test.loc[idx].to_dict()
    pred_label, prob = predict_transaction(best_model, txn, feature_names)
    match = "CORRECT" if pred_label == true_label else "WRONG"
    print(f"True label : {true_label}")
    print(f"Prediction : {pred_label}  (probability = {prob:.4f})  [{match}]")
    print()

---
## Final Summary

In [ ]:
print("="*60)
print("  CREDIT CARD FRAUD DETECTION – FINAL SUMMARY")
print("="*60)
print("\nModel Performance:")
print(df_res.to_string(index=False))
print(f"\nBest Model    : {best_name}")
print(f"F1-Score      : {best_row['F1']:.4f}")
print(f"ROC-AUC       : {best_row['ROC_AUC']:.4f}")
print(f"\nPlots saved to  : {PLOT_DIR}")
print(f"Model saved to  : {model_path}")
print("="*60)

---
## Tools Used
| Tool | Purpose |
|---|---|
| **Python** | Core programming language |
| **Pandas** | Data loading, manipulation, and DataFrame operations |
| **NumPy** | Numerical computations and array handling |
| **Scikit-learn** | ML models (Logistic Regression, Decision Tree, Random Forest), preprocessing, evaluation metrics |
| **Imbalanced-learn (SMOTE)** | Handling class imbalance via oversampling and undersampling |
| **Matplotlib** | Plotting charts, bar graphs, ROC curves, and visualisations |
| **Seaborn** | Heatmaps, confusion matrices, and styled plots |
| **Jupyter Notebook** | Interactive development environment for this analysis |

---
*CSC Project – Group 2 | Federal University of Technology Owerri*